# NYC ETA Engine -- Results & Validation

**Dev MAE: 253s** (28% below the 351s XGBoost baseline)

This notebook validates the submission end-to-end:
- Downloads pre-trained models from HuggingFace (no GPU needed)
- Scores on the challenge dev set
- Shows how the score improved at each stage
- Breaks down where the ensemble gains come from

**Runtime: ~15 min on Colab CPU** (50k predictions + analysis)

---
## Setup

In [ ]:
!git clone https://github.com/sarthakbiswas97/eta-engine.git 2>/dev/null
%cd eta-engine
!pip install -q -r requirements.txt huggingface_hub onnxruntime 2>/dev/null | tail -1
print("Dependencies installed.")

In [ ]:
# Download pre-trained models (6 MB total)
from huggingface_hub import hf_hub_download
import shutil, os

for f in ["model.pt", "lgbm_model.txt", "ft_model.pt", "ft_model.onnx", "ft_model.onnx.data", "ft_norm_params.npz"]:
    if not os.path.exists(f):
        shutil.copy2(hf_hub_download("sarthakbiswas/eta-engine", f), f)
print(f"Models ready: {sum(os.path.getsize(f) for f in os.listdir('.') if f.endswith(('.pt','.txt','.onnx','.npz'))) / 1e6:.1f} MB total")

# Download dev data + compute zone-pair stats
!python data/download_data.py 2>&1 | tail -1
!PYTHONPATH=. python -m features.zone_pair_stats 2>&1 | tail -1
print("Data ready.")

---
## 1. Does It Work? Score on Dev Set

The challenge baseline (XGBoost, 6 features) scores **351s** on dev.
Our 3-model ensemble should score **~253s**.

This runs the exact same `predict()` function that the grader calls.

In [ ]:
import sys, time
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
from predict import predict

# Score on 50k dev sample (same as grade.py)
dev = pd.read_parquet('data/dev.parquet').sample(n=50000, random_state=42)
records = dev[['pickup_zone', 'dropoff_zone', 'requested_at', 'passenger_count']].to_dict('records')

print(f"Scoring {len(records):,} dev rows...")
t0 = time.time()
preds = np.array([predict(r) for r in records])
elapsed = time.time() - t0

truth = dev['duration_seconds'].values
mae = np.mean(np.abs(preds - truth))
bias = np.mean(preds - truth)

print(f"\n{'='*45}")
print(f"  Dev MAE:    {mae:.1f}s")
print(f"  Baseline:   351.0s (XGBoost)")
print(f"  Improvement: {(1 - mae/351)*100:.0f}%")
print(f"  Bias:       {bias:+.1f}s")
print(f"  Latency:    {elapsed/len(records)*1000:.1f}ms per request")
print(f"  Total time: {elapsed:.0f}s for {len(records):,} rows")
print(f"{'='*45}")

---
## 2. How Did We Get Here? The Improvement Trajectory

Each bar is a real experiment with a specific motivation.
This wasn't random tuning -- each step was driven by analyzing what the previous model got wrong.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Experiment progression (actual measured values)
experiments = [
    ("Global Mean",          580, "Naive",     "Predict 989s for everything"),
    ("XGBoost\nBaseline",   351, "Baseline",  "Challenge baseline, 6 features"),
    ("Zone-Pair\nMedian",   297, "Features",  "Statistics only, zero ML"),
    ("Zone-Pair\nTime-Bucketed", 278, "Features", "6 time-of-day regimes"),
    ("NN v1",               272, "NN",        "Embeddings + L1 loss"),
    ("NN v2",               266, "NN",        "+ Huber loss + temporal stats"),
    ("NN v3",               265, "NN",        "+ Residual blocks"),
    ("NN v4b",              264, "NN",        "+ Lower dropout (diagnostic)"),
    ("NN+LGBM",             254, "Ensemble",  "LightGBM solves rare pairs"),
    ("NN+LGBM+FT",          253, "Ensemble",  "+ FT-Transformer diversity"),
]

names = [e[0] for e in experiments]
maes = [e[1] for e in experiments]
phases = [e[2] for e in experiments]
notes = [e[3] for e in experiments]

# Color by phase
colors = {'Naive': '#bdbdbd', 'Baseline': '#ef5350', 'Features': '#42a5f5', 'NN': '#66bb6a', 'Ensemble': '#ab47bc'}
bar_colors = [colors[p] for p in phases]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(range(len(names)), maes, color=bar_colors, edgecolor='white', linewidth=0.5)

# Add value labels
for i, (bar, m, note) in enumerate(zip(bars, maes, notes)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8, f"{m}s",
            ha='center', va='bottom', fontweight='bold', fontsize=10)
    if i >= 2:  # Skip first two (too high)
        ax.text(bar.get_x() + bar.get_width()/2, 230, note,
                ha='center', va='top', fontsize=7, color='#555', style='italic',
                rotation=45)

# Baseline reference line
ax.axhline(y=351, color='#ef5350', linestyle='--', alpha=0.5, linewidth=1)
ax.text(len(names)-0.5, 355, 'XGBoost baseline (351s)', ha='right', color='#ef5350', fontsize=9)

ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=9)
ax.set_ylabel('Dev MAE (seconds)', fontsize=12)
ax.set_title('ETA Prediction: Improvement Trajectory', fontsize=14, fontweight='bold')
ax.set_ylim(200, 620)

# Legend
legend_patches = [mpatches.Patch(color=c, label=l) for l, c in colors.items()]
ax.legend(handles=legend_patches, loc='upper right', fontsize=9)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f"Total improvement: {580}s -> {253}s ({(1-253/351)*100:.0f}% below baseline)")

---
## 3. NN Training Curves: How the Neural Net Learned

Three NN versions, each motivated by the previous one's failure mode.
All converge by epoch 3-4, then overfit -- the gap narrows with each version.

In [ ]:
# Actual training logs from Kaggle T4 GPU
epochs = [1, 2, 3, 4, 5, 6, 7, 8]
v1_mae = [858.7, 414.5, 275.2, 272.1, 273.9, 272.4, 274.6, 274.6]
v2_mae = [923.5, 394.5, 270.8, 270.3, 266.2, 269.0, 270.9, 270.4]
v3_mae = [300.8, 279.7, 272.3, 268.7, 264.5, 268.2, 269.3, 271.2]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Full view
ax1.plot(epochs, v1_mae, 'o-', label='v1: L1 loss, 19 features', color='#42a5f5', linewidth=2)
ax1.plot(epochs, v2_mae, 's-', label='v2: +Huber, +temporal stats', color='#66bb6a', linewidth=2)
ax1.plot(epochs, v3_mae, '^-', label='v3: +residual, +embeddings', color='#ab47bc', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Dev MAE (seconds)')
ax1.set_title('Training Convergence (full view)', fontweight='bold')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(250, 950)

# Zoomed view (epochs 3-8)
ax2.plot(epochs[2:], v1_mae[2:], 'o-', label='v1: 272.1s (epoch 4)', color='#42a5f5', linewidth=2)
ax2.plot(epochs[2:], v2_mae[2:], 's-', label='v2: 266.2s (epoch 5)', color='#66bb6a', linewidth=2)
ax2.plot(epochs[2:], v3_mae[2:], '^-', label='v3: 264.5s (epoch 5)', color='#ab47bc', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dev MAE (seconds)')
ax2.set_title('Zoomed: The Diminishing Returns', fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(260, 280)

# Annotate best epochs
ax2.annotate('v1 best', xy=(4, 272.1), fontsize=8, color='#42a5f5')
ax2.annotate('v2 best', xy=(5, 266.2), fontsize=8, color='#66bb6a')
ax2.annotate('v3 best', xy=(5, 264.5), fontsize=8, color='#ab47bc')

plt.suptitle('Neural Net Learning Curves (Kaggle T4 GPU, 37M rows)', fontsize=13)
plt.tight_layout()
plt.show()

print("Key insight: all versions plateau after epoch 5 and start overfitting.")
print("The gap between versions narrows: v1->v2 = 6s, v2->v3 = 1.7s.")
print("This confirmed the NN had hit a structural ceiling at ~264s.")

---
## 4. The Diagnostic That Changed the Strategy

Instead of tuning the NN further, we analyzed **where** the errors lived.
This analysis directly motivated adding LightGBM to the ensemble.

In [ ]:
# Rare-pair analysis (actual diagnostic results from scripts/diagnose.py)
categories = ['10k+\n(875k rows)', '1k-10k\n(286k)', '101-1k\n(47k)', '11-100\n(17k)', '1-10\n(5k)', 'Unseen\n(492)']
nn_maes = [251, 291, 356, 566, 747, 926]
nn_bias = [-42, -8, -23, -112, -243, -436]
avg_durations = [849, 1278, 1524, 2031, 2595, 2829]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# MAE by frequency
bar_colors = ['#66bb6a', '#66bb6a', '#ffb74d', '#ef5350', '#ef5350', '#b71c1c']
bars = ax1.bar(categories, nn_maes, color=bar_colors, edgecolor='white')
for bar, m in zip(bars, nn_maes):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15, f"{m}s",
            ha='center', fontweight='bold', fontsize=10)
ax1.set_ylabel('MAE (seconds)', fontsize=12)
ax1.set_title('NN Error by Zone-Pair Frequency', fontweight='bold')
ax1.set_ylim(0, 1050)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Bias by frequency
ax2.bar(categories, nn_bias, color=['#42a5f5' if b > -50 else '#ef5350' for b in nn_bias], edgecolor='white')
ax2.axhline(0, color='black', linewidth=0.5)
for i, b in enumerate(nn_bias):
    ax2.text(i, b - 30 if b < 0 else b + 10, f"{b:+d}s", ha='center', fontweight='bold', fontsize=10)
ax2.set_ylabel('Bias (seconds)', fontsize=12)
ax2.set_title('NN Underprediction Bias by Frequency', fontweight='bold')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle('The Rare-Pair Problem: Why the NN Ceiling Exists', fontsize=13)
plt.tight_layout()
plt.show()

print("Finding: the NN is near-optimal for common routes (251s MAE for 10k+ pairs).")
print("The error lives in the long tail -- rare pairs the NN hasn't seen enough of.")
print("Bias scales with rarity: -42s for common, -436s for unseen.")
print("")
print("This is why LightGBM was added: trees partition on zone IDs directly,")
print("no embedding interpolation needed. LGBM bias: -6s (vs NN's -106s).")

---
## 5. Ensemble Validation: Each Model's Contribution

The three models make different errors. Blending them reduces MAE by 8s
over the best single model.

In [ ]:
import torch
import lightgbm as lgb
from features.pipeline import FeaturePipeline
from model.architecture import ETAModel, ModelConfig
from model.ft_transformer import FTTransformer, FTConfig, FTConfigSmall
from model.dataset import create_dataloader
import gc

pipeline = FeaturePipeline.from_artifacts('data/zone_pair_stats/zone_pair_stats.pkl')
cat, cont, targets = pipeline.transform_dataframe(dev.copy())

# NN predictions
nn_cp = torch.load('model.pt', map_location='cpu', weights_only=False)
nn_model = ETAModel(ModelConfig(**nn_cp['model_config']))
nn_model.load_state_dict(nn_cp['model_state_dict']); nn_model.eval()
cont_nn = (cont - nn_cp['norm_params']['means']) / nn_cp['norm_params']['stds']
dummy = np.zeros(len(cat), dtype=np.float32)
loader = create_dataloader(cat, cont_nn, dummy, batch_size=16384, shuffle=False, num_workers=0, pin_memory=False)
nn_preds = np.concatenate([nn_model(pu, do, c).detach().cpu().numpy() for pu, do, c, _ in loader])
del nn_model, nn_cp; gc.collect()

# LGBM predictions
lgbm_model = lgb.Booster(model_file='lgbm_model.txt')
lgbm_preds = lgbm_model.predict(np.hstack([cat.astype(np.float32), cont]))
del lgbm_model; gc.collect()

# FT predictions
ft_cp = torch.load('ft_model.pt', map_location='cpu', weights_only=False)
ft_cfg = ft_cp['model_config']
ft_config = FTConfigSmall(**ft_cfg) if ft_cfg.get('d_token', 128) <= 96 else FTConfig(**ft_cfg)
ft_model = FTTransformer(ft_config)
ft_model.load_state_dict(ft_cp['model_state_dict']); ft_model.eval()
cont_ft = ((cont - ft_cp['norm_params']['means']) / ft_cp['norm_params']['stds']).astype(np.float32)
x_num = torch.from_numpy(cont_ft).float()
x_cat = torch.from_numpy(cat).long()
ft_preds = np.concatenate([ft_model(x_num[i:i+16384], x_cat[i:i+16384]).detach().cpu().numpy() for i in range(0, len(x_num), 16384)])
del ft_model, ft_cp; gc.collect()

# Ensemble
ensemble_preds = 0.5 * nn_preds + 0.3 * lgbm_preds + 0.2 * ft_preds

# Results table
print(f"{'Model':<20} {'MAE':>8} {'Bias':>8} {'Median AE':>10}")
print("-" * 50)
for name, p in [("NN (MLP)", nn_preds), ("LightGBM", lgbm_preds), ("FT-Transformer", ft_preds), ("Ensemble (0.5/0.3/0.2)", ensemble_preds)]:
    errors = np.abs(p - targets)
    print(f"{name:<20} {errors.mean():>8.1f} {(p-targets).mean():>+8.1f} {np.median(errors):>10.1f}")

In [ ]:
# Error distribution: why ensemble works
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

model_data = [
    ("NN (MLP) -- bias: -106s", nn_preds, '#42a5f5'),
    ("LightGBM -- bias: -6s", lgbm_preds, '#66bb6a'),
    ("FT-Transformer -- bias: -1s", ft_preds, '#ff7043'),
    ("Ensemble -- bias: -43s", ensemble_preds, '#ab47bc'),
]

for ax, (name, p, color) in zip(axes.flat, model_data):
    errors = p - targets
    mae = np.mean(np.abs(errors))
    ax.hist(errors, bins=100, range=(-1500, 1500), alpha=0.7, color=color, edgecolor='white')
    ax.axvline(0, color='black', linestyle='-', linewidth=0.5)
    ax.axvline(errors.mean(), color='red', linestyle='--', linewidth=1.5, label=f'bias={errors.mean():+.0f}s')
    ax.set_title(f'{name}\nMAE={mae:.0f}s', fontweight='bold', fontsize=11)
    ax.set_xlabel('Error (predicted - actual) seconds')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    ax.set_xlim(-1500, 1500)

plt.suptitle('Error Distributions: Each Model Fails Differently', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Error correlation
nn_errors = nn_preds - targets
lgbm_errors = lgbm_preds - targets
ft_errors = ft_preds - targets
print(f"Error correlations:")
print(f"  NN vs LGBM: {np.corrcoef(nn_errors, lgbm_errors)[0,1]:.3f}")
print(f"  NN vs FT:   {np.corrcoef(nn_errors, ft_errors)[0,1]:.3f}")
print(f"  LGBM vs FT: {np.corrcoef(lgbm_errors, ft_errors)[0,1]:.3f}")
print(f"\nCorrelation < 1.0 means the models make different errors.")
print(f"Blending decorrelated errors reduces MAE beyond any single model.")

---
## 6. Time-of-Day Behavior

The same route takes very different amounts of time depending on when you ride.
The model captures this from the time-bucketed zone-pair statistics.

In [ ]:
# Predict same route across 24 hours
routes = [
    (236, 237, "Midtown short trip"),
    (132, 138, "JFK area trip"),
    (48,  79,  "Cross-borough trip"),
]

fig, ax = plt.subplots(figsize=(12, 5))
colors_route = ['#2196F3', '#f44336', '#4CAF50']

for (pu, do, label), color in zip(routes, colors_route):
    hours = list(range(24))
    predictions = [predict({'pickup_zone': pu, 'dropoff_zone': do,
                           'requested_at': f'2024-02-14T{h:02d}:00:00',
                           'passenger_count': 1}) for h in hours]
    ax.plot(hours, [p/60 for p in predictions], 'o-', label=f'{label} ({pu}->{do})',
            color=color, linewidth=2, markersize=4)

ax.axvspan(7, 9, alpha=0.08, color='red', label='AM Rush')
ax.axvspan(16, 19, alpha=0.08, color='orange', label='PM Rush')
ax.axvspan(23, 24, alpha=0.05, color='blue')
ax.axvspan(0, 5, alpha=0.05, color='blue', label='Late Night')

ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Predicted Duration (minutes)', fontsize=12)
ax.set_title('Same Routes, Different Times: Model Captures Traffic Patterns', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels([f'{h}:00' for h in range(0, 24, 2)])
plt.tight_layout()
plt.show()

print("Late night trips are fastest (minimal traffic).")
print("Rush hour trips take 1.5-2.2x longer than late night.")
print("This comes from the 6 time-bucketed zone-pair statistics.")

---
## 7. What Failed (Honest Assessment)

Not everything worked. These failures were as important as the successes --
they informed the final design decisions.

In [ ]:
failures = [
    ["Hash buckets 16k -> 8k",     "Save params",          264, 277, "Hash embeddings are 47% of params but critical"],
    ["Remove month features",      "Cleaner signal",       264, 277, "Training needs seasonal signal for eval"],
    ["Log-target + Huber",         "Fix skewed targets",   264, 265, "Huber(300) in log-space = pure MSE"],
    ["LGBM on 37M rows",           "More data = better",   263, 267, "Outliers dilute tree splits"],
    ["Prediction rescaling",       "Fix variance collapse", 254, 253, "Only -0.8s, not worth overfitting risk"],
]

fig, ax = plt.subplots(figsize=(12, 5))

names_f = [f[0] for f in failures]
before = [f[2] for f in failures]
after = [f[3] for f in failures]
deltas = [a - b for a, b in zip(after, before)]

bar_colors_f = ['#ef5350' if d > 0 else '#bdbdbd' for d in deltas]
bars = ax.barh(names_f, deltas, color=bar_colors_f, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.5)

for i, (bar, d, note) in enumerate(zip(bars, deltas, [f[4] for f in failures])):
    x_pos = d + 0.5 if d > 0 else d - 0.5
    ax.text(max(d, 0) + 1, i, f"+{d}s  {note}" if d > 0 else f"{d}s  {note}",
            va='center', fontsize=9, color='#555')

ax.set_xlabel('MAE Change (seconds)', fontsize=12)
ax.set_title('Experiments That Didn\'t Help (red = regression)', fontsize=13, fontweight='bold')
ax.set_xlim(-5, 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## Summary

| Metric | Value |
|--------|-------|
| Dev MAE | **253s** |
| vs Baseline | **-28%** (351s -> 253s) |
| Models | MLP (560k) + LightGBM (81 trees) + FT-Transformer (169k) |
| Inference | ~4ms per request (limit: 200ms) |
| Docker | 1.4 GB (limit: 2.5 GB) |
| Total weights | 6.1 MB |
| Key insight | Diagnostic-driven: rare-pair analysis motivated the ensemble |

Pre-trained models: [huggingface.co/sarthakbiswas/eta-engine](https://huggingface.co/sarthakbiswas/eta-engine)

Full writeup: [README.md](https://github.com/sarthakbiswas97/eta-engine)